# 01 – Kom i gang med Slettix Analytics

Denne notebooken demonstrerer:
- Opprette en SparkSession koblet til den lokale Spark-clusteren
- Grunnleggende DataFrame-operasjoner
- Skrive og lese en Delta-tabell på MinIO


## 1. Opprett SparkSession

`spark-defaults.conf` er montert inn og konfigurerer Delta Lake og S3A/MinIO automatisk.  
Vi trenger kun å spesifisere master-URL-en.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("01-getting-started")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark versjon : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

## 2. Grunnleggende DataFrame-operasjoner

In [ ]:
from datetime import date

data = [
    (1, "Alice",   "engineering", 95000, date(2022, 3, 1)),
    (2, "Bob",     "marketing",   72000, date(2021, 7, 15)),
    (3, "Charlie", "engineering", 105000, date(2020, 1, 10)),
    (4, "Diana",   "marketing",   68000, date(2023, 5, 20)),
    (5, "Eve",     "engineering", 98000, date(2019, 11, 3)),
]

df = spark.createDataFrame(data, ["id", "name", "department", "salary", "hire_date"])
df.printSchema()
df.show()

In [ ]:
# Gjennomsnittlig lønn per avdeling
df.groupBy("department") \
  .agg(
      F.count("*").alias("antall"),
      F.avg("salary").alias("gj_snitt_lønn"),
      F.max("salary").alias("maks_lønn")
  ) \
  .orderBy("department") \
  .show()

## 3. Skriv til Delta-tabell på MinIO (Bronze-lag)

In [ ]:
TARGET_PATH = "s3a://bronze/employees"

df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(TARGET_PATH)

print(f"Skrev {df.count()} rader til {TARGET_PATH}")

## 4. Les tilbake fra Delta-tabell

In [ ]:
df_delta = spark.read.format("delta").load(TARGET_PATH)

print(f"Rader lest: {df_delta.count()}")
df_delta.show()

## 5. Inspiser Delta-tabellens historikk

Delta Lake loggfører alle operasjoner. Her ser vi versjonshistorikken.

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, TARGET_PATH)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

In [ ]:
spark.stop()